# A/B-тест: кнопка «Купить в 1 клик» на маркетплейсе

## Бизнес-контекст
Продуктовая команда маркетплейса hypothesizes, что замена стандартной корзины
на кнопку «Купить в 1 клик» упростит путь пользователя и повысит конверсию в покупку.

## Дизайн эксперимента
- **Группа A (контроль):** стандартная корзина
- **Группа B (тест):** кнопка «Купить в 1 клик»
- **Сплит:** 50/50, длительность 14 дней
- **Единица рандомизации:** пользователь (user_id)

## Гипотезы
- **H0:** конверсия в группе B равна конверсии в группе A (p_B = p_A)
- **H1:** конверсии различаются (p_B ≠ p_A), тест двусторонний (консервативный подход)

## Метрики
| Тип | Метрика | Определение |
|---|---|---|
| Primary | Conversion Rate | доля пользователей с покупкой (converted) |
| Secondary | AOV (средний чек) | медиана/среднее order_value среди купивших |
| Secondary | ARPU (выручка на пользователя) | среднее revenue по всем пользователям |
| Guardrail | session_duration | время сессии: не должно деградировать |

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

DATA_PATH = Path("../data/ab_test_data.csv")

## Шаг 1. Power analysis: сколько пользователей нужно для теста?

Задаём параметры дизайна:
- baseline-конверсия (группа A): 10%
- MDE (минимальный эффект, который бизнес хочет детектить): +1 п.п. (абсолютных)
- alpha = 0.05, power = 0.80

In [2]:
BASELINE_CONV = 0.10   # базовая конверсия в контроле
MDE = 0.01             # минимальный детектируемый эффект: +1 п.п.
ALPHA = 0.05           # уровень значимости (вероятность ошибки I рода)
POWER = 0.80           # мощность теста (1 - вероятность ошибки II рода)

# Эффект size для сравнения двух долей (метод Коэна через арксинус-преобразование)
effect_size = proportion_effectsize(BASELINE_CONV + MDE, BASELINE_CONV)

n_required = NormalIndPower().solve_power(
    effect_size=effect_size,
    alpha=ALPHA,
    power=POWER,
    ratio=1.0,
    alternative="two-sided",
)
n_required = int(np.ceil(n_required))
print(f"Минимальный размер выборки на группу: {n_required} пользователей")

Минимальный размер выборки на группу: 14745 пользователей


## Шаг 2. Генерация синтетических данных

Закладываем «истинные» эффекты (нам они известны, бизнесу — нет):
- Конверсия: A = 10.0%, B = 11.5% → **значимый эффект** (+1.5 п.п.)
- Средний чек: медиана A = 3000 ₽, B = 2900 ₽ → эффект **незначимый** (шум)
- Распределение чека: логнормальное → скошенное, с выбросами (как в реальном e-com)
- Guardrail (session_duration): одинаковый в обеих группах → эффект отсутствует

Добавим «грязь» как в реальном мире: аномальные чеки («киты») и потерянные суммы заказов (NaN).

In [3]:
SEED = 52                      # фиксируем seed для воспроизводимости
rng = np.random.default_rng(SEED)

N_PER_GROUP = 15000
TEST_DAYS = 14
START_DATE = "2026-08-01"

CONV_A, CONV_B = 0.100, 0.115          # истинные конверсии
AOV_MEDIAN_A, AOV_MEDIAN_B = 3000, 2900
SIGMA_LOG = 0.6                        # разброс логнормального распределения

dates = pd.date_range(START_DATE, periods=TEST_DAYS, freq="D")

def generate_group(group: str, conv: float, aov_median: float, n: int) -> pd.DataFrame:
    df = pd.DataFrame({
        "user_id": [f"{group}-{i:05d}" for i in range(n)],
        "group": group,
        "date": rng.choice(dates, size=n),
        "converted": rng.binomial(1, conv, size=n),
    })
    # Чек только у купивших: логнормальное распределение (скошение + выбросы)
    df["order_value"] = np.where(
        df["converted"] == 1,
        rng.lognormal(mean=np.log(aov_median), sigma=SIGMA_LOG, size=n),
        np.nan,
    ).round(2)
    # Guardrail-метрика: время сессии, сек (гамма-распределение, одинаковое в группах)
    df["session_duration"] = rng.gamma(shape=2.0, scale=90.0, size=n).round(1)
    return df

df = pd.concat(
    [generate_group("A", CONV_A, AOV_MEDIAN_A, N_PER_GROUP),
     generate_group("B", CONV_B, AOV_MEDIAN_B, N_PER_GROUP)],
    ignore_index=True,
)

In [4]:
# 1. «Киты»: 8 аномальных чеков 250k–900k ₽ (проверяем устойчивость тестов к выбросам)
buyers_idx = df.index[df["converted"] == 1]
whales_idx = rng.choice(buyers_idx, size=8, replace=False)
df.loc[whales_idx, "order_value"] = rng.uniform(250000, 900000, size=8).round(2)

# 2. Потерянные данные: ~0.3% заказов без суммы (NaN)
lost_idx = rng.choice(buyers_idx, size=int(len(buyers_idx) * 0.003), replace=False)
df.loc[lost_idx, "order_value"] = np.nan

# 3. Выручка на пользователя (ARPU): у некупивших = 0
df["revenue"] = df["order_value"].where(df["converted"] == 1, 0.0)

# Сохраняем
DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(DATA_PATH, index=False)
print(f"Сохранено: {DATA_PATH.resolve()}")
print(f"Строк: {len(df)}, столбцы: {list(df.columns)}")
df.head(10)

Сохранено: C:\Users\777\Desktop\a-b test project\data\ab_test_data.csv
Строк: 30000, столбцы: ['user_id', 'group', 'date', 'converted', 'order_value', 'session_duration', 'revenue']


,user_id,group,date,converted,order_value,session_duration,revenue
0,A-00000,A,2026-08-14,0,NaN,70.8,0.00
1,A-00001,A,2026-08-09,0,NaN,194.4,0.00
2,A-00002,A,2026-08-04,0,NaN,52.3,0.00
3,A-00003,A,2026-08-09,0,NaN,68.1,0.00
4,A-00004,A,2026-08-07,0,NaN,110.3,0.00
5,A-00005,A,2026-08-08,0,NaN,292.2,0.00
6,A-00006,A,2026-08-12,0,NaN,61.9,0.00
7,A-00007,A,2026-08-01,1,2523.47,399.3,2523.47
8,A-00008,A,2026-08-05,0,NaN,114.0,0.00
9,A-00009,A,2026-08-08,1,743.72,53.0,743.72


In [5]:
# Санити-чек: материализовались ли заложенные эффекты?
df.groupby("group").agg(
    users=("user_id", "count"),
    conversions=("converted", "sum"),
    conv_rate=("converted", "mean"),
    median_check=("order_value", "median"),
    mean_session_sec=("session_duration", "mean"),
).round(3)

,users,conversions,conv_rate,median_check,mean_session_sec
group,,,,,
A,15000,1464,0.098,3040.425,179.792
B,15000,1757,0.117,2990.230,179.785
